### Project on Train Simulation and Optimization

In [46]:
import numpy as np
import matplotlib.pyplot as plt

In [47]:
import numpy as np
import pandas as pd

stations = ["G", "L", "B", "Z"]

# Demand rates during peak periods [1000 passengers/h]
demand_peak = np.array(
    [
        [np.nan, 1.5, 2.2, 1.3],
        [np.nan, np.nan, 2.4, 1.1],
        [np.nan, np.nan, np.nan, 3.3],
        [np.nan, np.nan, np.nan, np.nan],
    ]
)

# Demand rates during off-peak periods [1000 passengers/h]
demand_offpeak = np.array(
    [
        [np.nan, 0.4, 0.3, 0.5],
        [np.nan, np.nan, 0.5, 0.3],
        [np.nan, np.nan, np.nan, 0.5],
        [np.nan, np.nan, np.nan, np.nan],
    ]
)

# Proportion of first class passengers during peak periods
first_class_ratio_peak = np.array(
    [
        [np.nan, 0.13, 0.15, 0.17],
        [np.nan, np.nan, 0.21, 0.18],
        [np.nan, np.nan, np.nan, 0.32],
        [np.nan, np.nan, np.nan, np.nan],
    ]
)

# Proportion of first class passengers during off-peak periods
first_class_ratio_offpeak = np.array(
    [
        [np.nan, 0.21, 0.18, 0.23],
        [np.nan, np.nan, 0.24, 0.23],
        [np.nan, np.nan, np.nan, 0.19],
        [np.nan, np.nan, np.nan, np.nan],
    ]
)

# Convert to pandas DataFrames
df_demand_peak = pd.DataFrame(demand_peak, index=stations, columns=stations)
df_demand_offpeak = pd.DataFrame(demand_offpeak, index=stations, columns=stations)
df_first_class_ratio_peak = pd.DataFrame(
    first_class_ratio_peak, index=stations, columns=stations
)
df_first_class_ratio_offpeak = pd.DataFrame(
    first_class_ratio_offpeak, index=stations, columns=stations
)

print("Demand peak:")
print(df_demand_peak)
print("\nDemand off-peak:")
print(df_demand_offpeak)
print("\nFirst class ratio peak:")
print(df_first_class_ratio_peak)
print("\nFirst class ratio off-peak:")
print(df_first_class_ratio_offpeak)

Demand peak:
    G    L    B    Z
G NaN  1.5  2.2  1.3
L NaN  NaN  2.4  1.1
B NaN  NaN  NaN  3.3
Z NaN  NaN  NaN  NaN

Demand off-peak:
    G    L    B    Z
G NaN  0.4  0.3  0.5
L NaN  NaN  0.5  0.3
B NaN  NaN  NaN  0.5
Z NaN  NaN  NaN  NaN

First class ratio peak:
    G     L     B     Z
G NaN  0.13  0.15  0.17
L NaN   NaN  0.21  0.18
B NaN   NaN   NaN  0.32
Z NaN   NaN   NaN   NaN

First class ratio off-peak:
    G     L     B     Z
G NaN  0.21  0.18  0.23
L NaN   NaN  0.24  0.23
B NaN   NaN   NaN  0.19
Z NaN   NaN   NaN   NaN


In [48]:
from dataclasses import dataclass
from enum import StrEnum


class CustomerClass(StrEnum):
    FIRST = "First Class"
    SECOND = "Second Class"


class Station(StrEnum):
    """G->L->B->Z"""

    G = "Geneva"
    L = "Lausanne"
    B = "Bern"
    Z = "Zurich"

    def possible_destinations(self):
        if self == Station.G:
            return [Station.L, Station.B, Station.Z]
        elif self == Station.L:
            return [Station.B, Station.Z]
        elif self == Station.B:
            return [Station.Z]
        else:
            return []

    def travel_time_next(self):
        if self == Station.G:
            return 0
        elif self == Station.L:
            return 40
        elif self == Station.B:
            return 70
        elif self == Station.Z:
            return 60


@dataclass
class TrainType:
    name: str
    first_class_carriages: int
    second_class_carriages: int


@dataclass
class Train:
    start_time: int
    train_type: TrainType
    seats_first_class: int = 0
    seats_second_class: int = 0
    current_station: Station = Station.G

    @property
    def first_class_seats(self):
        return self.train_type.first_class_carriages * 300

    @property
    def second_class_seats(self):
        return self.train_type.second_class_carriages * 500

    def next(self):
        if self.current_station == Station.G:
            self.current_station = Station.L
        elif self.current_station == Station.L:
            self.current_station = Station.B
        elif self.current_station == Station.B:
            self.current_station = Station.Z
        else:
            raise ValueError("Train is already at the last station.")

## Events

In [49]:
# not using inheritance since it is simpler
# just use an event protocol
from typing import Protocol


class Event(Protocol):
    """Base class for events"""


@dataclass
class CustomerArrival(Event):
    """Arrival of customer at the station where his journey starts."""

    time: int
    start: Station
    destination: Station
    customer_class: CustomerClass


@dataclass
class CustomerBoarding(Event):
    """Customer boards the train."""

    time: int


@dataclass
class TrainArrival(Event):
    """Train arrives at a station."""

    time: int
    station: Station
    train_type: TrainType = None


normal_train = TrainType(
    name="Normal", first_class_carriages=3, second_class_carriages=1
)


def generate_simple_timetable(freq: int = 20):
    train_events = []
    for t in range(freq, 24 * 60, freq):
        train_journey = []
        for station in Station:
            travel_time = station.travel_time_next()
            # print(f"Train starting at {t} will arrive at {station} after {travel_time} minutes.")
            train_journey.append(
                TrainArrival(
                    time=t + travel_time, station=station, train_type=normal_train
                )
            )
        train_events.extend(train_journey)
    return train_events


In [50]:
simple_timetable = generate_simple_timetable(freq=20)

for event in simple_timetable[:6]:
    print(event)

# sort by time
print()
print("Sorted timetable:")
simple_timetable.sort(key=lambda x: x.time)
for event in simple_timetable[:6]:
    print(event)

TrainArrival(time=20, station=<Station.G: 'Geneva'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))
TrainArrival(time=60, station=<Station.L: 'Lausanne'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))
TrainArrival(time=90, station=<Station.B: 'Bern'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))
TrainArrival(time=80, station=<Station.Z: 'Zurich'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))
TrainArrival(time=40, station=<Station.G: 'Geneva'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))
TrainArrival(time=80, station=<Station.L: 'Lausanne'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1))

Sorted timetable:
TrainArrival(time=20, station=<Station.G: 'Geneva'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carria

In [51]:
@dataclass
class Scenario:
    """all times in minutes"""

    initial_events: list[Event] = None


first_scenario = Scenario(initial_events=generate_simple_timetable(freq=20))
first_scenario
# TODO: add initial generation event for person

Scenario(initial_events=[TrainArrival(time=20, station=<Station.G: 'Geneva'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=60, station=<Station.L: 'Lausanne'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=90, station=<Station.B: 'Bern'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=80, station=<Station.Z: 'Zurich'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=40, station=<Station.G: 'Geneva'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=80, station=<Station.L: 'Lausanne'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_class_carriages=1)), TrainArrival(time=110, station=<Station.B: 'Bern'>, train_type=TrainType(name='Normal', first_class_carriages=3, second_c

In [52]:
event_list = []
for station in Station:
    for destination in station.possible_destinations():
        print(f"Generating arrival event for train from {station} to {destination}.")
        # ArrivalEvent = generate(current_time, station, destination)
        # event_list.append(ArrivalEvent)

        # if isinstance(event) == "arrival-person":
        # generate new person

Generating arrival event for train from Geneva to Lausanne.
Generating arrival event for train from Geneva to Bern.
Generating arrival event for train from Geneva to Zurich.
Generating arrival event for train from Lausanne to Bern.
Generating arrival event for train from Lausanne to Zurich.
Generating arrival event for train from Bern to Zurich.
